PDF 파일기반 질의응답 챗봇 (랭체인, 그라디오, Upstage)

In [1]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS


c:\myclass\ai_langchain\mylangchin-uv-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 1.PDF 로드
loader = PyPDFLoader("../data/tutorial-korean.pdf")
documents = loader.load()
print(len(documents))

39


In [4]:
# 2. 텍스트 분할
#text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200, 
    chunk_overlap=0,
    separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""]
)

#texts = text_splitter.split_documents(documents)
texts = loader.load_and_split(text_splitter=text_splitter)
print(len(texts))

284


In [5]:
from langchain_upstage import UpstageEmbeddings

embeddings = UpstageEmbeddings(model="solar-embedding-1-large")

# 3.1 FAISS 벡터 저장소 생성
vector_store = FAISS.from_documents(texts, embeddings)
vector_store.save_local("../db/faiss_db")

# 3.2 벡터 저장소에서 검색하기
# search_type="similarity" 옵션은 retrieval 객체에서 유사성 검색을 사용하여 질문 vector와 가장 유사한 문장 vector를 선택함
# search_kwargs 옵션은 vector 저장소에서 유사한 텍스트 덩어리 k(=6)개를 검색해 Prompt로 보내려는 것을 의미함
retriever = vector_store.as_retriever(
    search_type="similarity", 
    search_kwargs={"k": 6}
)

In [6]:
from langchain_upstage import ChatUpstage

llm = ChatUpstage(
        model="solar-pro3",
        base_url="https://api.upstage.ai/v1",
        temperature=0.5
)
print(llm.model_name)

solar-pro3


In [7]:
from langchain_core.prompts import PromptTemplate

# 한국어 최적화 프롬프트
prompt_template = """
당신은 BlueJ 프로그래밍 환경 전문가입니다. 
아래 문서 내용을 바탕으로 정확하고 친절한 답변을 제공해주세요.

문서 내용:
{context}

질문: {question}

답변 규칙:
1. 문서 내용만을 근거로 답변하세요
2. 단계별 설명이 필요하면 순서대로 작성하세요  
3. 구체적인 메뉴명, 버튼명을 포함하세요
4. 문서에 없는 정보는 "문서에서 찾을 수 없습니다"라고 하세요

답변:"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)
print(" 프롬프트 설정 완료")
print(prompt)

 프롬프트 설정 완료
input_variables=['context', 'question'] input_types={} partial_variables={} template='\n당신은 BlueJ 프로그래밍 환경 전문가입니다. \n아래 문서 내용을 바탕으로 정확하고 친절한 답변을 제공해주세요.\n\n문서 내용:\n{context}\n\n질문: {question}\n\n답변 규칙:\n1. 문서 내용만을 근거로 답변하세요\n2. 단계별 설명이 필요하면 순서대로 작성하세요  \n3. 구체적인 메뉴명, 버튼명을 포함하세요\n4. 문서에 없는 정보는 "문서에서 찾을 수 없습니다"라고 하세요\n\n답변:'


#### RAG 체인(LCEL)의 데이터 흐름

```python
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
```

`rag_chain.invoke(query)`로 넘긴 **같은 `query`가 dict의 각 값에 동시에 전달**됩니다.

```text
rag_chain.invoke(query)
   │
   ├─ "context":  retriever | format_docs
   │      query -> retriever -> [Document, ...] -> format_docs -> "문서1

문서2..." (str)
   └─ "question": RunnablePassthrough()
          query -> 가공 없이 그대로 -> "코드패드는 ..." (str)
   │
   ▼  {"context": "...", "question": "..."}
prompt -> llm -> StrOutputParser() -> 답변 문자열
```

| 단계 | 하는 일 |
|---|---|
| `retriever` | 질문(`query`)과 유사한 문서를 벡터DB에서 검색해 `Document` 리스트를 반환합니다. |
| `format_docs` | `Document` 리스트를 `page_content`만 이어 붙인 **하나의 문자열**로 바꿉니다. |
| `RunnablePassthrough()` | 입력받은 `query`를 **그대로** 통과시킵니다. |
| `prompt` | dict의 `context`, `question` 값을 프롬프트의 `{context}`, `{question}` 자리에 채웁니다. |
| `llm` | 완성된 프롬프트로 답변을 생성합니다. (`AIMessage`) |
| `StrOutputParser()` | `AIMessage`에서 문자열만 꺼냅니다. |

* **dict는 자동으로 `RunnableParallel`이 됩니다.** 체인 안의 `{...}`는 각 값을 병렬로 실행해 결과 dict를 만드는 `RunnableParallel`로 자동 변환되므로, 직접 쓰지 않아도 같은 동작입니다.
* **dict의 키와 프롬프트 변수 이름은 같아야 합니다.** `"context"`, `"question"` 키가 `prompt`의 `{context}`, `{question}`에 그대로 대응하며, 이름이 다르면 변수 누락 오류가 납니다.
* **한계**: `format_docs`를 거치면 문서 객체가 문자열이 되어 사라지므로, 이 체인의 반환값에는 출처(metadata)가 없습니다. 출처가 필요하면 `retriever`를 따로 호출하거나, 아래 `rag_chain_with_sources`처럼 체인을 바꿔야 합니다.

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# LCEL RAG chain using RunnablePassthrough
def format_docs(docs):
    """Format retrieved docs as context string"""
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("LCEL RAG chain 설정 완료")

# Invoke with question (returns answer string)
query = "코드패드는 무엇이고 어떻게 사용하나요?"
query_result = rag_chain.invoke(query)
print(query_result)

# To get source documents, access retriever separately or modify chain
source_docs = retriever.invoke(query)
print("Source documents:", [doc.metadata for doc in source_docs])

In [ ]:
query = "애플릿은 무엇이고 어떻게 사용하나요?"
query_result = rag_chain.invoke(query)
print(type(query_result))
print(query_result)

#### 챗봇용 RAG 체인

앞의 `rag_chain`은 **답변 문자열만** 반환해서 출처(metadata)를 알 수 없습니다. `retriever`를 따로 호출하면 검색이 2번 일어나므로, **검색을 한 번만** 하고 문서와 답변을 함께 반환하도록 바꿉니다. (`llm`, `prompt`, `retriever`, `format_docs`는 위에서 만든 것을 재사용)

```python
answer_chain = (
    RunnableLambda(lambda x: {"context": format_docs(x["docs"]), "question": x["question"]})
    | prompt | llm | StrOutputParser()
)

rag_chain_with_sources = (
    RunnableParallel(docs=retriever, question=RunnablePassthrough())
    .assign(answer=answer_chain)
)
```

**데이터 흐름**

```text
질문 ─► RunnableParallel ─► {"docs": [Document, ...], "question": "질문"}
                                      │  (.assign 이 이 dict 를 answer_chain 의 입력으로 전달)
                                      ▼
                  answer_chain 의 RunnableLambda 에서  x = 위 dict
                  x -> {"context": format_docs(x["docs"]), "question": x["question"]}
                                      ▼
                  prompt -> llm -> StrOutputParser -> "답변"
                                      ▼
최종 반환: {"docs": [...], "question": "질문", "answer": "답변"}
```

* **`docs`, `question` 키**는 `RunnableParallel(docs=..., question=...)`의 인자 이름이 그대로 키가 됩니다. 이 dict가 `answer_chain`의 `x`로 전달되므로, `answer_chain`은 이 dict 입력을 전제로 만든 체인입니다.
* **`RunnableLambda`는 변환기**입니다. 입력 키(`docs`, `question`)를 `prompt`의 변수 이름(`{context}`, `{question}`)에 맞게 바꿉니다. `docs`는 `format_docs`로 문자열로 바꿔 `context`로, `question`은 값을 그대로 옮깁니다.
* **`.assign(answer=...)`**는 기존 dict를 유지한 채 `answer` 키만 추가하므로, 반환값에 답변(`answer`)과 근거 문서(`docs`)가 함께 담깁니다. 답변에 쓰인 문서와 출처가 항상 일치합니다.

In [9]:
from langchain_core.runnables import RunnableLambda, RunnableParallel

# RunnableParallel(...)이 만든 결과 dict(docs, question 키)가 .assign()을 통해 answer_chain으로 넘어가고, 첫 단계인 RunnableLambda의 x가 됩니다.

# 답변 체인: {"docs", "question"} -> 문서 포맷팅 -> prompt -> llm -> 문자열
answer_chain = (
    RunnableLambda(lambda x: {"context": format_docs(x["docs"]), "question": x["question"]})
    | prompt
    | llm
    | StrOutputParser()
)

# 검색은 한 번만 수행하고, 같은 문서로 답변을 만들어 {"docs", "question", "answer"} 를 반환
rag_chain_with_sources = (
    RunnableParallel(docs=retriever, question=RunnablePassthrough())
    .assign(answer=answer_chain)
)

result = rag_chain_with_sources.invoke("코드패드는 무엇이고 어떻게 사용하나요?")
print(result["answer"])
print("Source documents:", [doc.metadata for doc in result["docs"]])

**코드패드란?**  
BlueJ 코드패드는 자바 코드(표현식과 명령문)의 일부분을 쉽고 빠르게 평가할 수 있는 기능입니다. 이를 통해 프로그램 코드의 의미를 상세히 조사하거나 구문을 예증하고 시험할 수 있습니다.

---

### 코드패드 사용 방법

1. **코드패드 보기 열기**  
   - 메인 화면에서 **보기 메뉴(View)** 를 선택합니다.  
   - **코드패드 보기(Show Code Pad)** 를 클릭하여 코드패드 창을 엽니다.  

2. **코드 입력**  
   - 코드패드 영역에 **표현식 또는 명령문** 을 입력합니다.  
   - 키보드의 **Enter** 키를 누르면 입력된 내용이 라인 단위로 평가되어 결과가 화면에 나타납니다.  

3. **입력 이력 활용**  
   - 이전에 입력한 내용은 코드패드에서 자동으로 저장됩니다.  
   - **상하 화살표** 를 사용하여 이전에 입력한 내용을 다시 불러오거나 수정할 수 있습니다.  

4. **결과 확인**  
   - 각 입력 라인의 평가 결과는 코드패드 창에 즉시 표시되며, 이를 통해 코드의 동작을 실시간으로 확인할 수 있습니다.  

---

이와 같이 코드패드는 간단한 자바 표현식을 빠르게 테스트하고 디버깅하는 데 유용하게 사용됩니다.
Source documents: [{'producer': 'Acrobat Distiller with ezUniHFT', 'creator': 'PScript5.dll Version 5.2', 'creationdate': '2005-04-26T15:21:34+09:00', 'moddate': '2005-04-26T15:21:34+09:00', 'author': 'Owner', 'title': '<426C75654AC7D1B1DBC6A9C5E4B8AEBEF3B9AEBCAD283230292E687770>', 'source': '../data/tutorial-korean.pdf', 'total_pages': 39, 'page': 1, 'page_l

##### 서버 실행/종료 헬퍼 함수

| 함수 | 하는 일 | 언제 쓰나요? |
|---|---|---|
| `launch_demo(demo)` | 켜져 있는 서버를 먼저 끄고 `demo`를 새로 실행합니다. | 챗봇을 띄울 때. 셀을 다시 실행해도 포트 충돌이 없습니다. |
| `close_demo()` | 실행 중인 Gradio 서버를 모두 끕니다. | 실습을 마칠 때. 포트(7860)가 반납됩니다. |
| `is_port_open()` | 포트를 누군가 쓰고 있으면 `True`, 아니면 `False`를 반환합니다. | 서버가 꺼졌는지 확인할 때. |

In [ ]:
import socket
import gradio as gr

GRADIO_HOST = "127.0.0.1"   # 내 컴퓨터에서만 접속 가능한 주소 (외부 접속 차단)
GRADIO_PORT = 7860          # 웹 서버가 사용할 포트 번호 (브라우저 주소: http://127.0.0.1:7860)

def is_port_open(port: int = GRADIO_PORT, host: str = GRADIO_HOST) -> bool:
    """해당 포트에서 서버가 응답하면 True(점유 중), 아니면 False(해제됨)를 반환한다."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.5)                       # 응답이 없으면 0.5초 뒤 포기
        return s.connect_ex((host, port)) == 0  # 접속 성공(0)이면 누군가 포트를 쓰는 중

def close_demo() -> None:
    """실행 중인 모든 Gradio 서버를 종료하여 포트를 해제한다."""
    gr.close_all()

def launch_demo(demo: gr.Blocks, port: int = GRADIO_PORT) -> None:
    """이전 서버를 종료한 뒤 demo를 실행한다. 셀을 다시 실행해도 포트 충돌이 없다."""
    close_demo()  # 이미 켜진 서버가 있으면 먼저 끈다 (포트 충돌 방지)
    # prevent_thread_lock=True: 서버를 켠 뒤 셀 실행을 끝낸다 (False면 서버가 꺼질 때까지 셀이 멈춤)
    demo.launch(server_name=GRADIO_HOST, server_port=port, prevent_thread_lock=True)

In [ ]:
ERROR_PREFIX = "⚠️ 답변 생성 중 오류가 발생했습니다: "  # 오류 메시지 접두어


def format_sources(docs) -> str:
    """검색된 문서의 출처(파일명, 페이지)를 번호 목록 문자열로 만든다."""
    return "\n".join(
        f'[{i + 1}] {doc.metadata.get("source", "Unknown")} (Page {doc.metadata.get("page", "N/A")})'
        for i, doc in enumerate(docs)
    )


def chat_respond(message, chat_history):
    """질문을 받아 RAG 체인으로 답변하고 채팅 기록에 추가한다. (전역 rag_chain_with_sources 재사용)"""
    try:
        result = rag_chain_with_sources.invoke(message)
        bot_message = f"{result['answer']}\n\n{format_sources(result['docs'])}"
    except Exception as e:  # API 키 누락, 네트워크 오류 등을 채팅창에 표시
        bot_message = f"{ERROR_PREFIX}{e}"

    # Gradio 채팅 기록 형식(dict)에 맞춰 저장
    chat_history.append({"role": "user", "content": message})
    chat_history.append({"role": "assistant", "content": bot_message})

    # 입력 창 초기화 및 갱신된 채팅 기록 반환
    return "", chat_history


# Gradio UI 생성 및 실행
with gr.Blocks() as demo:
    chatbot = gr.Chatbot(label="채팅창")       # 채팅 메시지를 표시하는 컴포넌트
    msg = gr.Textbox(label="입력")             # 사용자 입력 텍스트 박스
    clear = gr.Button("초기화")                # 채팅 기록 초기화 버튼

    # 입력창에서 Enter -> chat_respond 실행 -> 입력창 비우고 채팅 기록 갱신
    msg.submit(chat_respond, [msg, chatbot], [msg, chatbot])
    # 초기화 버튼 클릭 시 채팅 기록 비우기
    clear.click(lambda: [], None, chatbot, queue=False)

# Gradio 앱 실행 (이전 서버를 먼저 끄고 실행하므로 셀을 다시 실행해도 포트 충돌이 없음)
launch_demo(demo)

In [ ]:
close_demo()
print("포트 점유 여부:", is_port_open())   # False면 해제 완료